# CNN Training Notebook

### 0. Import Library & Dataset

In [1]:
# Standard libraries
import os
import sys
import itertools
import time
import json
import glob

from pathlib import Path

# Data Processing
import numpy as np
import pandas as pd

# Visualisasi
import matplotlib.pyplot as plt

# Evaluasi Model
from sklearn.metrics import f1_score, classification_report

# Deep Learning
import tensorflow as tf

print('TF: ', tf.__version__)
print('GPU: ', tf.config.list_physical_devices('GPU'))


[TensorFlow DLL Diagnostic] Analyzing: c:\Users\Ardy\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow\python\_pywrap_tensorflow_internal.pyd
[Error] Failed to load _pywrap_tensorflow_common.dll: INITIALIZATION FAILED (0x45A) - The DLL's DllMain returned false.
    Hint: This often happens if your CPU lacks required instructions (like AVX/AVX2)
    or if the Microsoft Visual C++ Redistributable is outdated/missing.


ImportError: Traceback (most recent call last):
  File "c:\Users\Ardy\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 74, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

In [ ]:
src = Path(os.getcwd()) # Path on this file
while not (src / 'src').exists() and src != src.parent:
    src = src.parent

sys.path.insert(0, str(src))
os.chdir(src)

In [ ]:
DATA_DIR = Path('data/intel')
MODEL_DIR = Path('models/cnn')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR  = DATA_DIR / 'test'

### 1. Load Dataset

In [ ]:
CLASSES = []

for iterdir in TRAIN_DIR.iterdir():
    if iterdir.is_dir():
        CLASSES.append(iterdir.name)

CLASSES.sort()
print('Classes are ', CLASSES)

# 80:20 Split untuk Train & Test dataset
# Define Variables
IMG_SIZE = 150
BATCH_SIZE = 64
EPOCS = 20
NUM_CLASSES = 6

# 1. Train
train_split_ = tf.keras.utils.image_dataset_from_directory(TRAIN_DIR, 
                                                           image_size = (IMG_SIZE, IMG_SIZE),
                                                           batch_size = BATCH_SIZE,
                                                           label_mode = 'int',
                                                           seed = 42,
                                                           class_names = CLASSES,
                                                           validation_split  = 0.2,
                                                           subset = 'training',
                                                           )

# 2. Validation
validate_split_ = tf.keras.utils.image_dataset_from_directory(TRAIN_DIR,
                                                              image_size = (IMG_SIZE, IMG_SIZE),
                                                              batch_size = BATCH_SIZE,
                                                              label_mode = 'int',
                                                              seed = 42,
                                                              class_names = CLASSES,
                                                              validation_split = 0.2,
                                                              subset = 'validation',
                                                              )

# 3. Test
test_split_ = tf.keras.utils.image_dataset_from_directiory(TEST_DIR,
                                                           image_size = (IMG_SIZE, IMG_SIZE),
                                                           batch_size = BATCH_SIZE,
                                                           label_mode = 'int',
                                                           shuffle = False,
                                                           class_names = CLASSES,
                                                           )

norm = tf.keras.layers.Rescaling(1./255)
train_ds = train_split_.map(lambda x,y: (norm(x), y)).cache().prefetch(tf.data.AUTOTUNE)
val_ds = validate_split_.map(lambda x,y: (norm(x), y)).cache().prefetch(tf.data.AUTOTUNE)
test_ds = validate_split_.map(lambda x,y: (norm(x), y)).cache().prefetch(tf.data.AUTOTUNE)

label_list = []
for gambar, label in test_ds:
    label_numpy = label.numpy()

    label_list.append(label_numpy)

y_test = np.concatenate(label_list)